In [ ]:
!pip install requests pandas


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.8 MB/s  0:00:02 eta 0:00:01
Using cached certifi-2026.5.20-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pandas]2m6/7 [pandas]_normalizer]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
import requests
import json
import pandas as pd


def fetch_fund_data(scheme_code):
    print(f'Fetching data for scheme code: {scheme_code}')
    url=f'https://api.mfapi.in/mf/{scheme_code}'
    response= requests.get(url)

    if response.status_code == 200:
        data= response.json()
        
        meta= data.get('meta',{})
        fund_name=meta.get('scheme_name')

        nav_history= data.get('data' , [])
        df= pd.DataFrame(nav_history)
        if not df.empty:
            # Clean and format the data
            df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
            df['nav'] = pd.to_numeric(df['nav'])
            df['scheme_code'] = scheme_code
            df['scheme_name'] = fund_name
            
            # Sort chronologically
            df = df.sort_values('date').reset_index(drop=True)
            print(f"✅ Successfully fetched {len(df)} days of historical data for {fund_name}")
            return df
    
    print(f"❌ Failed to fetch data.")
    return None

# Test the function with a specific fund code
test_df = fetch_fund_data(119062)

test_df

Fetching data for scheme code: 119062
✅ Successfully fetched 3308 days of historical data for HDFC Hybrid Equity Fund - Growth Option - Direct Plan


,date,nav,scheme_code,scheme_name
0,2013-01-01,28.957,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
1,2013-01-02,29.227,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
2,2013-01-03,29.316,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
3,2013-01-04,29.383,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
4,2013-01-07,29.350,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
...,...,...,...,...
3303,2026-06-08,118.675,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
3304,2026-06-09,119.440,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
3305,2026-06-10,119.350,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...
3306,2026-06-11,119.044,119062,HDFC Hybrid Equity Fund - Growth Option - Dire...


In [10]:
import requests
import pandas as pd
import os

print("Connecting to AMFI to download the master database... please wait.")

# 1. Hit the master AMFI endpoint that contains EVERY fund
url = "https://api.mfapi.in/mf"
response = requests.get(url)

if response.status_code == 200:
    # 2. Parse the massive JSON response
    master_data = response.json()
    df_master = pd.DataFrame(master_data)
    
    # 3. Ensure your target folder exists
    os.makedirs('../data/raw', exist_ok=True)
    
    # 4. Save all 40,000+ rows directly to a CSV
    file_path = '../data/raw/amfi_master_list.csv'
    df_master.to_csv(file_path, index=False)
    
    print(f"✅ BOOM. Successfully saved {len(df_master)} mutual fund codes to {file_path}!")
    
    # Show the first few rows just to prove it worked
    display(df_master.head())
else:
    print(f"❌ Failed to connect to AMFI. Error Code: {response.status_code}")

Connecting to AMFI to download the master database... please wait.
✅ BOOM. Successfully saved 37635 mutual fund codes to ../data/raw/amfi_master_list.csv!


,schemeCode,schemeName,isinGrowth,isinDivReinvestment
0,100027,Grindlays Super Saver Income Fund-GSSIF-Half Y...,NaN,NaN
1,100028,Grindlays Super Saver Income Fund-GSSIF-Quater...,NaN,NaN
2,100029,Grindlays Super Saver Income Fund-GSSIF-Growth,NaN,NaN
3,100030,Grindlays Super Saver Income Fund-GSSIF-Annual...,NaN,NaN
4,100031,Grindlays Super Saver Income Fund-GSSIF - ST-D...,NaN,NaN


In [9]:
# Read the paired funds file you will generate from the master list
universe_df = pd.read_csv('data/raw/fund_pairs.csv')

# Automatically convert the rows into the exact list-of-dictionaries format
target_funds = universe_df.to_dict('records')

all_funds_data= []

for fund in target_funds:
    print(f"--- Processing {fund['name']} ---")

    ## Direct Plan
    direct_df= fetch_fund_data(fund['direct_code'])
    if direct_df is  not None:
        direct_df['plan_type']='Direct'
        all_funds_data.append(direct_df)

    ## Regular Plan
    regular_df=fetch_fund_data(fund['regular_code'])
    if regular_df is not None:
        regular_df['plan_type']='Regular'
        all_funds_data.append(regular_df)


master_df=pd.concat(all_funds_data , ignore_index=True)
    
start_date='2019-01-01'
end_date='2023-12-31'

master_df= master_df[(master_df['date'] >= start_date )& (master_df['date']<= end_date)]


print(f"\nFinal dataset ready! Total rows for 2019-2023: {len(master_df)}")

master_df.sample(5)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/fund_pairs.csv'